# Suite EXT — Schema extension

A schema is extended by adding members to it: properties and adjacencies to an object schema, properties to a
relation, branches to a union (docs/SCHEMA_EXTENSION.md). `store.extend` extends the schemas a store holds;
`Builders.extend` extends the framework, the meta-schemas every store holds, once per process. Anything but added
members is refused. The cases that extend the framework come last: a notebook's cases share its process, and so the
framework's extensions.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Plain, Validators
from support import raises

Text, Int = B.Meta.OfStr, B.Meta.OfInt
store = B.Store()
Knows = store.Schemas.Relation(lambda r: r.name("crm.Knows").links([lambda t: t.name("a"), lambda t: t.name("b")]))
Person = store.Schemas.Object(lambda o: (
    o
    .name("crm.Person")
    .ref(True)
    .properties([lambda t: t.name("name").type([Text])])
))
Customer = store.Schemas.Object(lambda o: (
    o
    .name("crm.Customer")
    .ref(True)
    .refines([Person])
    .properties([lambda t: t.name("account").type([Text])])
))
Id = store.Schemas.Union(lambda u: u.name("crm.Id").branches([lambda t: t.name("n").type([Int])]))
EXTENDS = "an extension only adds members to"


## EXT-01 · `store.extend` adds members to the schemas a store holds; their objects and their refinements gain them

In [ ]:
ada = store.crm.Person(lambda p: p.name("Ada"))  # built before the extension
store.extend(lambda m: (
    m.Schemas.Object(Person, lambda o: o.properties([lambda t: t.name("age").type([Int])])),
    m.Schemas.Relation(Knows, lambda r: r.properties([lambda t: t.name("since").type([Int])])),
    m.Schemas.Union(Id, lambda u: u.branches([lambda t: t.name("s").type([Text])])),
    m.Schemas.Object(lambda o: o.name("crm.Note").properties([lambda t: t.name("text").type([Text])])),  # built: its own
))
assert list(Person.properties) == ["name", "age"] and list(Customer.properties) == ["name", "age", "account"]  # inherited
assert list(Knows.entry.properties) == ["since"] and [b.name for b in Id.branches] == ["n", "s"]
assert ada.age.get() is None and store.crm.Person(ada, lambda p: p.age(36)).age.get() == 36  # absent until written
assert store.crm.Customer(lambda c: c.name("Bo").age(7).account("A")).age.get() == 7
assert store.registered("crm.Note").validate() == [] and store.extend(lambda m: m) is store


## EXT-02 · The extension check refuses anything but added members, saying what the extension would change

In [ ]:
for extension, says in [
    (lambda m: m.Schemas.Object(Person, lambda o: o.description("A person")), "'crm.Person': it changes 'description'"),
    (lambda m: m.Schemas.Object(Person, lambda o: o.ref(False)), "'crm.Person': it changes 'ref'"),
    (lambda m: m.Schemas.Object(Person, lambda o: o.properties(lambda adv: adv.remove(lambda e: e.name("name")))),
     "'crm.Person': it removes or replaces what 'properties' holds"),
    (lambda m: m.Schemas.Object(Person, lambda o: o.properties([lambda t: t.name("name").type([Int])])),  # retyped
     "'crm.Person': it removes or replaces what 'properties' holds"),
    (lambda m: m.Schemas.Relation(Knows, lambda r: r.links([lambda t: t.name("c")])),
     "'crm.Knows': it adds to 'links', which an extension cannot"),
    (lambda m: m.Schemas.Object(Person, lambda o: o.properties([lambda t: t.name("account").type([Text])])),
     "'crm.Person': it adds 'account', which 'crm.Customer' declares"),
    (lambda m: m.Schemas.Object(Customer, lambda o: o.properties([lambda t: t.name("name").type([Text])])),
     "'crm.Customer': it adds 'name', which it inherits"),
    (lambda m: m.Schemas.Object(Customer, lambda o: o.adjacencies([lambda t: t.name("name").relation([Knows]).me("a")])),
     "'crm.Customer': it adds 'name', which it inherits"),
]:
    with raises(TypeError, match=f"{EXTENDS} {says}"):
        store.extend(extension)
assert list(Person.properties) == ["name", "age"] and Person.ref and Person.description is None  # refused: not written
store.Schemas.Object(Person, lambda o: o.description("A person"))  # outside an extension, a store's own schema changes
assert Person.description == "A person"


## EXT-03 · A framework schema, which every store holds, changes only through `Builders.extend`

In [ ]:
for change in (lambda: store.Schemas.Object(B.Meta.Scalar, lambda o: o.description("x")),
               lambda: store.extend(lambda m: m.Schemas.Object(B.Meta.Scalar, lambda o: o.properties([
                   lambda t: t.name("unit").type([Text])])))):
    with raises(TypeError, match="'Schemas.Scalar' is a framework schema, which every store holds: extend it with Builders.extend"):
        change()
assert "unit" not in B.Meta.Scalar.properties
with raises(TypeError, match="an extension is named by non-empty text, got ''"):
    B.extend("", lambda m: m)


## EXT-04 · `Builders.extend` extends the framework once per process: every store holds what it built, and a meta-schema's new members are its schemas' builders'

In [ ]:
early = B.Store()  # made before the extension


def domains(m):
    Domain = m.Schemas.Object(lambda o: (
        o
        .name("Expressions.Framework.Domain")
        .ref(True)
        .properties([lambda t: t.name("label").type([Text])])
    ))
    HasDomain = m.Schemas.Relation(lambda r: (
        r
        .name("Expressions.Framework.HasDomain")
        .links([lambda t: t.name("schema"), lambda t: t.name("domain")])
        .uniques("schema")
    ))
    m.Schemas.Object(Domain, lambda o: o.adjacencies([lambda t: t.name("schemas").relation([HasDomain]).me("domain")]))
    m.Schemas.Object(B.OfScalar.Schema, lambda o: (
        o
        .properties([lambda t: t.name("unit").type([Text])])
        .adjacencies([lambda t: t.name("domain").relation([HasDomain]).me("schema")])
    ))


B.extend("test.domains", domains)
B.extend("test.domains", domains)  # applied: nothing again
late = B.Store()
for held in (early, late):
    Domain = held.registered("Expressions.Framework.Domain")
    assert held.framework(Domain) and held.framework(held.registered("Expressions.Framework.HasDomain"))
assert "domain" in B.OfScalar.Schema.adjacencies and "unit" in B.OfScalar.Schema.properties
Meters = late.Schemas.Scalar(lambda s: s.name("Meters").token("float").unit("m"))
assert B.OfScalar.Builder(Meters, False, late).unit.get() == "m"
Feet = B.OfScalar.Builder(Meters, store=late).name("Feet").clone()
assert B.OfScalar.Builder(Feet, False, late).unit.get() == "m"  # a clone keeps it
with raises(TypeError, match="an extension's adjacency 'trio' is to a relation of two links and no properties, got 'T3'"):
    B.extend("test.trio", lambda m: m.Schemas.Object(B.OfScalar.Schema, lambda o: o.adjacencies([
        lambda t: t.name("trio").relation([m.Schemas.Relation(lambda r: r.name("T3").links([
            lambda t: t.name("a"), lambda t: t.name("b"), lambda t: t.name("c")]))]).me("a")])))


## EXT-05 · An extension's relation links schema objects and data: one entry, held by both, written from either end and once in a snapshot

In [ ]:
s = B.Store()
int8 = s.Expressions.Framework.Domain(lambda d: d.label("int8"))
Byte = s.Schemas.Scalar(lambda x: x.name("Byte").token("int").domain([int8]))  # from the schema's end
assert [e.schema is Byte for e in int8.schemas] == [True]  # seen from the data's end
int16 = s.Expressions.Framework.Domain(lambda d: d.label("int16").schemas(lambda a: a.insert(lambda e: e.schema(Byte))))
assert [e.domain.label.get() for e in B.OfScalar.Builder(Byte, False, s).domain] == ["int8", "int16"]
assert Validators.Validate(s).Reachable(s.registered("Expressions.Framework.Domain"), int8) == []
log = Plain.ToPlain(s).Snapshot()
assert sum("insert" in mutation and "HasDomain" in str(mutation) for mutation in log) == 2  # each entry once
back = B.Store()
Plain.FromPlain(back).Snapshot(log)
assert len(list(B.OfScalar.Builder(back.registered("Byte"), False, back).domain)) == 2 and Plain.ToPlain(back).Snapshot() == log
s.Schemas.Scalar(Byte, lambda x: x.domain(lambda a: a.remove(lambda e: e.domain(int8))))
assert list(int8.schemas) == [] and len(list(int16.schemas)) == 1
with raises(TypeError, match="proxies can only link proxy instances, or schema objects their store holds"):
    s.Expressions.Framework.Domain(lambda d: d.schemas(lambda a: a.insert(lambda e: e.schema(B.Store().Schemas.Scalar(lambda x: x)))))
